In [1]:
import pandas as pd
from collections import defaultdict

class TablaHashSIPA:
    def __init__(self):
        # El diccionario actuará como nuestra Tabla Hash con complejidad O(1)
        self.tabla = defaultdict(list)
        self.total_reportes = 0
        self.reportes_recurrentes = 0
        
    def insertar_reporte(self, folio, celda_geo, fecha, estatus):
        self.total_reportes += 1
        
        # Si la celda ya existe en la memoria, detectamos presión social (reporte repetido)
        if celda_geo in self.tabla:
            self.reportes_recurrentes += 1
            
        # Agregamos el folio al casillero correspondiente a su coordenada
        self.tabla[celda_geo].append({
            'folio': folio,
            'fecha': fecha,
            'estatus': estatus
        })
        
    def buscar_bache(self, celda_geo):
        # Búsqueda instantánea O(1) sin importar si hay 35,000 o 1 millón de registros
        return self.tabla.get(celda_geo, [])
        
    def obtener_metricas(self):
        baches_unicos = len(self.tabla)
        return {
            "Total de Quejas Recibidas": self.total_reportes,
            "Baches Físicos Únicos": baches_unicos,
            "Quejas Duplicadas (Presión Social)": self.reportes_recurrentes
        }

# 1. Instanciamos el motor algorítmico
motor_hash = TablaHashSIPA()

# 2. Cargamos los datos limpios de la Fase 1
df_baches = pd.read_csv('sipa_etl_procesado.csv')

print("Inyectando registros en la Tabla Hash...")
# 3. Poblamos la estructura de datos
for index, fila in df_baches.iterrows():
    motor_hash.insertar_reporte(
        folio=fila['id_folio'],
        celda_geo=fila['celda_geo'],
        fecha=fila['fecha_solicitud'],
        estatus=fila['estatus']
    )

# 4. Imprimimos el resultado de la deduplicación
print("\n--- RESULTADOS DEL MOTOR DE DEDUPLICACIÓN ---")
metricas = motor_hash.obtener_metricas()
for metrica, valor in metricas.items():
    print(f"{metrica}: {valor:,}")

Inyectando registros en la Tabla Hash...

--- RESULTADOS DEL MOTOR DE DEDUPLICACIÓN ---
Total de Quejas Recibidas: 35,743
Baches Físicos Únicos: 35,274
Quejas Duplicadas (Presión Social): 469


In [2]:
import heapq
import pandas as pd

class MaxHeapSIPA:
    def __init__(self):
        # Esta lista actuará como nuestro árbol binario
        self.arbol = []
        # Fecha de corte simulada para calcular antigüedades (cierre de tu dataset)
        self.fecha_corte = pd.to_datetime('2024-12-31') 
        
    def calcular_ipu(self, lista_reportes):
        presion_social = len(lista_reportes)
        
        # Encontramos la fecha del reporte original (el más viejo en esa celda)
        fechas = [pd.to_datetime(r['fecha']) for r in lista_reportes if pd.notnull(r['fecha'])]
        if not fechas:
            return 0, 0
            
        fecha_mas_vieja = min(fechas)
        dias_antiguedad = (self.fecha_corte - fecha_mas_vieja).days
        if dias_antiguedad < 0: dias_antiguedad = 0
        
        # Algoritmo de calificación
        ipu = (dias_antiguedad * 1) + (presion_social * 30)
        return ipu, dias_antiguedad
        
    def inyectar_baches(self, tabla_hash_diccionario):
        print("Calculando IPU y construyendo el árbol binario...")
        for celda_geo, reportes in tabla_hash_diccionario.items():
            ipu, dias = self.calcular_ipu(reportes)
            
            if ipu > 0:
                # El signo negativo empuja el IPU más grande a la cima del árbol (Max-Heap)
                heapq.heappush(self.arbol, (-ipu, celda_geo, len(reportes), dias))
                
    def extraer_top_urgencias(self, cantidad=5):
        top_urgencias = []
        for _ in range(cantidad):
            if self.arbol:
                # heappop saca SIEMPRE la raíz del árbol (el elemento más urgente) en O(log N)
                bache = heapq.heappop(self.arbol)
                top_urgencias.append({
                    'IPU_Score': -bache[0], # Regresamos el IPU a positivo para leerlo
                    'Celda_Geo': bache[1],
                    'Quejas_Ciudadanas': bache[2],
                    'Dias_Espera': bache[3]
                })
        return top_urgencias

# 1. Instanciamos el Max-Heap
motor_prioridad = MaxHeapSIPA()

# 2. Le inyectamos los datos directamente desde tu Tabla Hash anterior
motor_prioridad.inyectar_baches(motor_hash.tabla)

# 3. Extraemos el Top 5 de baches más críticos de toda la CDMX
top_5_criticos = motor_prioridad.extraer_top_urgencias(5)

print("\n--- TOP 5 BACHES MÁS CRÍTICOS (LISTOS PARA CUADRILLA) ---")
df_top = pd.DataFrame(top_5_criticos)
print(df_top.to_string(index=False))

Calculando IPU y construyendo el árbol binario...

--- TOP 5 BACHES MÁS CRÍTICOS (LISTOS PARA CUADRILLA) ---
 IPU_Score        Celda_Geo  Quejas_Ciudadanas  Dias_Espera
      2119 19.4617_-99.1777                  3         2029
      2117 19.2582_-99.1191                  3         2027
      2107 19.4849_-99.1769                  3         2017
      2096 19.4304_-99.1689                  2         2036
      2092 19.4547_-99.0965                  2         2032


In [1]:
import pandas as pd
import numpy as np
import osmnx as ox
import geopandas as gpd
from shapely.geometry import Point
import scipy.integrate as integrate
import warnings

# Apagamos las advertencias visuales de pandas para mantener la terminal limpia
warnings.filterwarnings('ignore')

print("1. Cargando la base de datos completa de baches (SIN FILTROS)...")
# Quitamos el .head(100) para procesar los +35,000 registros
df_baches = pd.read_csv('sipa_etl_procesado.csv').dropna(subset=['latitud', 'longitud'])
geometria = [Point(xy) for xy in zip(df_baches['longitud'], df_baches['latitud'])]

# Proyectamos al sistema UTM (Metros) oficial del centro de México
gdf_baches = gpd.GeoDataFrame(df_baches, geometry=geometria, crs="EPSG:4326").to_crs("EPSG:32614")

print("2. Descargando la topología vial completa de la CDMX (Esto tomará tiempo)...")
# Usamos <Expansion term="graph_from_place">Función de OSMnx que descarga todas las vías de un polígono administrativo oficial.</Expansion>
grafo_vial = ox.graph_from_place('Ciudad de México, México', network_type='drive')
_, calles = ox.graph_to_gdfs(grafo_vial)
calles = calles.to_crs("EPSG:32614")

print("3. Ejecutando Cruce Espacial Masivo y limpiando duplicados de intersecciones...")
baches_con_calles = gpd.sjoin_nearest(gdf_baches, calles, how="left", distance_col="distancia_m")
baches_con_calles = baches_con_calles.drop_duplicates(subset=['id_folio'])

diccionario_aforo = {
    'motorway': 80000, 'trunk': 60000, 'primary': 35000, 
    'secondary': 15000, 'tertiary': 5000, 'residential': 1500, 'unclassified': 1000
}

def limpiar_tipo_via(tipo):
    if isinstance(tipo, list): return tipo[0]
    return tipo if pd.notnull(tipo) else 'residential'

baches_con_calles['tipo_via'] = baches_con_calles['highway'].apply(limpiar_tipo_via)
baches_con_calles['aforo_estimado'] = baches_con_calles['tipo_via'].map(diccionario_aforo).fillna(1500)

print("4. Integrando Motores de Cálculo Integral y APU a toda la base...")
def calcular_apu_bacheo(volumen_m3):
    costo_materiales = 2550.00 * volumen_m3 
    costo_mo = (3800 / 8) * volumen_m3
    costo_maq = (3200 / 8) * volumen_m3
    costo_directo = costo_materiales + costo_mo + costo_maq
    precio_unitario_total = costo_directo * 1.25 # 25% de sobrecosto legal
    return round(precio_unitario_total, 2)

def evaluar_riesgo(row):
    dias = (pd.to_datetime('2024-12-31') - pd.to_datetime(row['fecha_solicitud'])).days
    dias = max(1, dias)
    aforo = row['aforo_estimado']
    
    def degradacion(t):
        return (aforo * 0.00000015) * (1 + (0.001 * t))
    
    volumen, _ = integrate.quad(degradacion, 0, dias)
    volumen = round(volumen, 4)
    
    COSTO_INDEMNIZACION = 15000
    prob_accidente = min(0.95, (volumen * 0.1) + (aforo * 0.00001))
    pasivo = round(prob_accidente * COSTO_INDEMNIZACION, 2)
    costo_rep = calcular_apu_bacheo(volumen)
    
    return pd.Series([volumen, costo_rep, pasivo])

baches_con_calles[['volumen_m3', 'costo_reparacion', 'pasivo_NIF_C9']] = baches_con_calles.apply(evaluar_riesgo, axis=1)

print("5. Exportando el producto final para el Frontend...")
columnas_finales = ['id_folio', 'latitud', 'longitud', 'alcaldia', 'colonia', 'fecha_solicitud', 
                    'tipo_via', 'aforo_estimado', 'volumen_m3', 'costo_reparacion', 'pasivo_NIF_C9']
df_exportacion = baches_con_calles[columnas_finales]

# Guardamos el CSV final que alimentará tu página web HTML/JS
df_exportacion.to_csv('sipa_final_completo.csv', index=False)
print("¡ÉXITO! Backend terminado. Archivo 'sipa_final_completo.csv' creado correctamente.")

1. Cargando la base de datos completa de baches (SIN FILTROS)...
2. Descargando la topología vial completa de la CDMX (Esto tomará tiempo)...
3. Ejecutando Cruce Espacial Masivo y limpiando duplicados de intersecciones...
4. Integrando Motores de Cálculo Integral y APU a toda la base...
5. Exportando el producto final para el Frontend...


KeyError: "['colonia'] not in index"

In [2]:
print("5. Exportando el producto final para el Frontend...")

# 1. Extraemos usando el nombre real que viene en tus datos (colonia_datos)
columnas_finales = ['id_folio', 'latitud', 'longitud', 'alcaldia', 'colonia_datos', 'fecha_solicitud', 
                    'tipo_via', 'aforo_estimado', 'volumen_m3', 'costo_reparacion', 'pasivo_NIF_C9']

df_exportacion = baches_con_calles[columnas_finales].copy()

# 2. Renombramos la columna a "colonia" para que el frontend tenga un nombre limpio y estándar
df_exportacion = df_exportacion.rename(columns={'colonia_datos': 'colonia'})

# 3. Exportamos el CSV final
df_exportacion.to_csv('sipa_final_completo.csv', index=False)
print("¡ÉXITO! Backend terminado. Archivo 'sipa_final_completo.csv' exportado con todas las columnas.")

5. Exportando el producto final para el Frontend...
¡ÉXITO! Backend terminado. Archivo 'sipa_final_completo.csv' exportado con todas las columnas.
